In [9]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_key=os.getenv("GROQ_API_KEY")
jina_key=os.getenv("JINA_API_KEY")


# imports 
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
file_path=os.path.join("data","hr_policy.txt")

In [7]:
# data ingestion
# page content and metadata
loader=TextLoader(file_path=file_path,encoding="utf-8")
documents=loader.load()
len(documents[0].page_content)


3946

In [11]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks=text_splitter.split_documents(documents)
len(chunks)

12

In [12]:
from langchain_community.embeddings import JinaEmbeddings

embeddings_model=JinaEmbeddings(model_name="jina-embeddings-v2-base-en")



In [ ]:
# store data in vector db
from langchain_community.vectorstores import FAISS
vector_store=FAISS.from_documents(chunks,embeddings_model)


In [15]:
test="how many sick leaves we get per month"
top_matches=vector_store.similarity_search(test,k=4)

for i,match in enumerate (top_matches,start=1):
    print(f"---match {i}----")
    print(match.page_content)
    print()

---match 1----
## 2. Leave Policy
- Employees receive 18 paid leaves annually.
- Employees receive 10 sick leaves annually.
- Employees must request planned leave at least 3 days in advance.
- Emergency leave must be communicated to the reporting manager.
- Unused annual leaves can be carried forward up to 5 days.
- Leave approval depends on manager approval.

---match 2----
## 5. Salary Policy
- Salaries are paid on the last working day of every month.
- Salary details are mentioned in the employment offer letter.
- Employees receive monthly payslips.
- Salary-related issues should be reported to the HR department.
- Applicable statutory deductions are made from salaries.

---match 3----
## 4. Attendance Policy
- Employees must mark attendance using the company attendance system.
- Late arrivals must be reported to the manager.
- Repeated unapproved absences may result in disciplinary action.
- Employees must inform their manager if they cannot attend work.

---match 4----
## 8. Resig

In [25]:
# data reterival
from langchain_groq import ChatGroq
llm=ChatGroq(model="openai/gpt-oss-20b")

In [28]:
from langchain.agents import create_agent

In [33]:
retriver=vector_store.as_retriever(search_kwargs={"k":3})
def search(question):
    """returns matching chunks"""
    matching_chunks=retriver.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

In [39]:
from langchain.agents import create_agent
hr_assistant=create_agent(
    model=llm,
    tools=[search],
    system_prompt="""you are friendly hr assistant always use search_hr_policy to look up facts if ans is not in search results say idk instead of guessing"""
)

In [42]:
response=hr_assistant.invoke(
    {"messages":[{"role":"user","content":"how many sick leaves do i get"}]}
    
)
print(response["messages"][-1].content)

You’re entitled to **10 paid sick leaves per year**. If you need more or have any other questions about how to use them, just let me know!
